In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(PROCESSED_DIR / "train_events_v1.parquet")
valid = pd.read_parquet(PROCESSED_DIR / "validation_events_v1.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_events_v1.parquet")

feature_columns = pd.read_csv(
    PROCESSED_DIR / "feature_list_v1.csv"
)["feature_name"].tolist()

TARGET = "reversal"
CATEGORICAL_FEATURES = ["sector"]
NUMERIC_FEATURES = feature_columns
MODEL_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Train / validation / test:", len(train), len(valid), len(test))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)

Train / validation / test: 11288 1771 1785
Numeric features: 22
Categorical features: ['sector']


In [2]:
X_train = train[MODEL_COLUMNS].copy()
y_train = train[TARGET].copy()

X_valid = valid[MODEL_COLUMNS].copy()
y_valid = valid[TARGET].copy()

X_test = test[MODEL_COLUMNS].copy()
y_test = test[TARGET].copy()

print("Shapes:", X_train.shape, X_valid.shape, X_test.shape)
print("\nTraining class balance:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

Shapes: (11288, 23) (1771, 23) (1785, 23)

Training class balance:
reversal
0    72.74
1    27.26
Name: proportion, dtype: float64


In [3]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            NUMERIC_FEATURES
        ),
        (
            "sector",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)

X_train_encoded = xgb_preprocessor.fit_transform(X_train)
X_valid_encoded = xgb_preprocessor.transform(X_valid)
X_test_encoded = xgb_preprocessor.transform(X_test)

print("Encoded shapes:")
print("Train:", X_train_encoded.shape)
print("Validation:", X_valid_encoded.shape)
print("Test:", X_test_encoded.shape)

Encoded shapes:
Train: (11288, 33)
Validation: (1771, 33)
Test: (1785, 33)


In [4]:
def evaluate_probabilities(y_true, probabilities, threshold=0.50, model_name="XGBoost"):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": round(accuracy_score(y_true, predictions), 4),
        "precision": round(precision_score(y_true, predictions, zero_division=0), 4),
        "recall": round(recall_score(y_true, predictions, zero_division=0), 4),
        "f1": round(f1_score(y_true, predictions, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_true, probabilities), 4),
        "pr_auc": round(average_precision_score(y_true, probabilities), 4),
        "predicted_reversals": int(predictions.sum())
    }

In [5]:
xgb_model_v1 = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=10,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=5.0,
    reg_alpha=0.10,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_model_v1.fit(
    X_train_encoded,
    y_train
)

xgb_valid_probabilities_v1 = xgb_model_v1.predict_proba(
    X_valid_encoded
)[:, 1]

xgb_v1_results = evaluate_probabilities(
    y_valid,
    xgb_valid_probabilities_v1,
    threshold=0.50,
    model_name="XGBoost_v1"
)

pd.DataFrame([xgb_v1_results])

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_reversals
0,XGBoost_v1,0.5,0.8024,0.0,0.0,0.0,0.5858,0.2838,0


In [6]:
rf_summary = pd.read_csv(
    PROCESSED_DIR / "random_forest_selected_config_v1.csv"
)

comparison_v1 = pd.DataFrame([
    {
        "model": "Random Forest",
        "validation_pr_auc": rf_summary.loc[0, "validation_pr_auc"],
        "validation_roc_auc": rf_summary.loc[0, "validation_roc_auc"],
        "selected_threshold": rf_summary.loc[0, "selected_threshold"],
        "validation_precision_at_selected_threshold": rf_summary.loc[
            0, "validation_precision"
        ],
        "validation_recall_at_selected_threshold": rf_summary.loc[
            0, "validation_recall"
        ],
        "validation_f1_at_selected_threshold": rf_summary.loc[
            0, "validation_f1"
        ],
    },
    {
        "model": "XGBoost_v1",
        "validation_pr_auc": xgb_v1_results["pr_auc"],
        "validation_roc_auc": xgb_v1_results["roc_auc"],
        "selected_threshold": 0.50,
        "validation_precision_at_selected_threshold": xgb_v1_results[
            "precision"
        ],
        "validation_recall_at_selected_threshold": xgb_v1_results[
            "recall"
        ],
        "validation_f1_at_selected_threshold": xgb_v1_results["f1"],
    }
])

comparison_v1

,model,validation_pr_auc,validation_roc_auc,selected_threshold,validation_precision_at_selected_threshold,validation_recall_at_selected_threshold,validation_f1_at_selected_threshold
0,Random Forest,0.2974,0.5986,0.4,0.2491,0.5829,0.349
1,XGBoost_v1,0.2838,0.5858,0.5,0.0000,0.0000,0.000


In [7]:
xgb_threshold_rows = []

for threshold in np.arange(0.20, 0.71, 0.05):
    row = evaluate_probabilities(
        y_valid,
        xgb_valid_probabilities_v1,
        threshold=float(threshold),
        model_name="XGBoost_v1"
    )
    xgb_threshold_rows.append(row)

xgb_threshold_results_v1 = pd.DataFrame(xgb_threshold_rows)

xgb_threshold_results_v1[
    ["precision", "recall", "f1"]
] = xgb_threshold_results_v1[
    ["precision", "recall", "f1"]
].round(4)

xgb_threshold_results_v1[
    ["threshold", "predicted_reversals", "precision", "recall", "f1"]
]

,threshold,predicted_reversals,precision,recall,f1
0,0.20,834,0.2434,0.5800,0.3429
1,0.25,375,0.2960,0.3171,0.3062
2,0.30,162,0.3765,0.1743,0.2383
3,0.35,60,0.4000,0.0686,0.1171
4,0.40,9,0.6667,0.0171,0.0334
5,0.45,1,1.0000,0.0029,0.0057
6,0.50,0,0.0000,0.0000,0.0000
7,0.55,0,0.0000,0.0000,0.0000
8,0.60,0,0.0000,0.0000,0.0000
9,0.65,0,0.0000,0.0000,0.0000


In [8]:
xgb_best_threshold_row_v1 = xgb_threshold_results_v1.loc[
    xgb_threshold_results_v1["f1"].idxmax()
]

xgb_best_threshold_v1 = float(
    xgb_best_threshold_row_v1["threshold"]
)

print("Best XGBoost validation threshold by F1:", xgb_best_threshold_v1)
print(xgb_best_threshold_row_v1)

Best XGBoost validation threshold by F1: 0.2
model                  XGBoost_v1
threshold                     0.2
accuracy                   0.5607
precision                  0.2434
recall                       0.58
f1                         0.3429
roc_auc                    0.5858
pr_auc                     0.2838
predicted_reversals           834
Name: 0, dtype: object


In [9]:
xgb_best_predictions_v1 = (
    xgb_valid_probabilities_v1 >= xgb_best_threshold_v1
).astype(int)

print(classification_report(
    y_valid,
    xgb_best_predictions_v1,
    target_names=["No reversal", "Reversal"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_valid, xgb_best_predictions_v1))

              precision    recall  f1-score   support

 No reversal       0.84      0.56      0.67      1421
    Reversal       0.24      0.58      0.34       350

    accuracy                           0.56      1771
   macro avg       0.54      0.57      0.51      1771
weighted avg       0.72      0.56      0.61      1771

Confusion matrix:
[[790 631]
 [147 203]]


In [10]:
xgb_model_v2 = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_model_v2.fit(X_train_encoded, y_train)

xgb_valid_probabilities_v2 = xgb_model_v2.predict_proba(
    X_valid_encoded
)[:, 1]

xgb_v2_default_results = evaluate_probabilities(
    y_valid,
    xgb_valid_probabilities_v2,
    threshold=0.50,
    model_name="XGBoost_v2"
)

pd.DataFrame([xgb_v2_default_results])

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_reversals
0,XGBoost_v2,0.5,0.8024,0.5,0.0086,0.0169,0.5658,0.2565,6


In [11]:
xgb_v2_threshold_rows = []

for threshold in np.arange(0.10, 0.61, 0.025):
    row = evaluate_probabilities(
        y_valid,
        xgb_valid_probabilities_v2,
        threshold=float(threshold),
        model_name="XGBoost_v2"
    )
    xgb_v2_threshold_rows.append(row)

xgb_v2_threshold_results = pd.DataFrame(xgb_v2_threshold_rows)

xgb_v2_threshold_results[
    ["precision", "recall", "f1"]
] = xgb_v2_threshold_results[
    ["precision", "recall", "f1"]
].round(4)

xgb_v2_threshold_results[
    ["threshold", "predicted_reversals", "precision", "recall", "f1"]
]

,threshold,predicted_reversals,precision,recall,f1
0,0.100,1602,0.2010,0.9200,0.3299
1,0.125,1397,0.2033,0.8114,0.3251
2,0.150,1203,0.2111,0.7257,0.3271
3,0.175,978,0.2178,0.6086,0.3208
4,0.200,750,0.2373,0.5086,0.3236
5,0.225,580,0.2655,0.4400,0.3312
6,0.250,409,0.2738,0.3200,0.2951
7,0.275,310,0.2742,0.2429,0.2576
8,0.300,214,0.3084,0.1886,0.2340
9,0.325,145,0.3448,0.1429,0.2020


In [12]:
xgb_best_threshold_row_v2 = xgb_v2_threshold_results.loc[
    xgb_v2_threshold_results["f1"].idxmax()
]

xgb_best_threshold_v2 = float(
    xgb_best_threshold_row_v2["threshold"]
)

print("Best XGBoost v2 threshold by F1:", xgb_best_threshold_v2)
print(xgb_best_threshold_row_v2)

print("\nValidation PR-AUC:", round(
    average_precision_score(y_valid, xgb_valid_probabilities_v2),
    4
))
print("Validation ROC-AUC:", round(
    roc_auc_score(y_valid, xgb_valid_probabilities_v2),
    4
))

Best XGBoost v2 threshold by F1: 0.22499999999999998
model                  XGBoost_v2
threshold                   0.225
accuracy                   0.6488
precision                  0.2655
recall                       0.44
f1                         0.3312
roc_auc                    0.5658
pr_auc                     0.2565
predicted_reversals           580
Name: 5, dtype: object

Validation PR-AUC: 0.2565
Validation ROC-AUC: 0.5658


In [13]:
xgb_comparison_summary = pd.DataFrame([
    {
        "model": "Random Forest",
        "validation_pr_auc": 0.2974,
        "validation_roc_auc": 0.5986,
        "best_f1": 0.3490,
        "selected_threshold": 0.40,
        "precision_at_selected_threshold": 0.2491,
        "recall_at_selected_threshold": 0.5829,
        "selected_as_final": True
    },
    {
        "model": "XGBoost_v1",
        "validation_pr_auc": 0.2838,
        "validation_roc_auc": 0.5858,
        "best_f1": 0.3429,
        "selected_threshold": 0.20,
        "precision_at_selected_threshold": 0.2434,
        "recall_at_selected_threshold": 0.5800,
        "selected_as_final": False
    },
    {
        "model": "XGBoost_v2",
        "validation_pr_auc": 0.2565,
        "validation_roc_auc": 0.5658,
        "best_f1": 0.3312,
        "selected_threshold": 0.225,
        "precision_at_selected_threshold": 0.2655,
        "recall_at_selected_threshold": 0.4400,
        "selected_as_final": False
    }
])

xgb_comparison_summary.to_csv(
    PROCESSED_DIR / "model_comparison_validation_v1.csv",
    index=False
)

xgb_comparison_summary

,model,validation_pr_auc,validation_roc_auc,best_f1,selected_threshold,precision_at_selected_threshold,recall_at_selected_threshold,selected_as_final
0,Random Forest,0.2974,0.5986,0.3490,0.400,0.2491,0.5829,True
1,XGBoost_v1,0.2838,0.5858,0.3429,0.200,0.2434,0.5800,False
2,XGBoost_v2,0.2565,0.5658,0.3312,0.225,0.2655,0.4400,False
